## Estimating cortical surface area from ROIs

This interactive notebook demonstrates how one can use the predicted retinotopic maps derived from deepRetinotopy to estimate cortical surface area dedicated to representing different portions of the visual field or specific ROIs. Although deepRetinotopy does not perform visual area segmentation automatically, it offers an algorithm to estimate [visual field sign maps](https://felenitaribeiro.github.io/deepRetinotopy_TheToolbox/retinotopic_mapping/sign_maps/) to guide manual delineation. Because this feature is not yet fully optimised in the current toolbox, it will be an optional part of this notebook. For more details about the toolbox, please check our [repository](https://github.com/felenitaribeiro/deepRetinotopy_TheToolbox) and [webpage](https://felenitaribeiro.github.io/deepRetinotopy_TheToolbox/). 
Requirements:

1. FreeSurfer directory;
2. HCP "fs_LR-deformed_to-fsaverage" surfaces, available [here](https://github.com/Washington-University/HCPpipelines/tree/master/global/templates/standard_mesh_atlases/resample_fsaverage).
3. Predicted polar angle and eccentricity maps.

### Citation and Resources
#### Tools included in this workflow
__deepretinotopy__:
- Ribeiro, F. L. et al. (2025). Predicting functional topography of the human visual cortex from cortical anatomy at scale. [bioRxiv](https://www.biorxiv.org/content/10.1101/2025.11.27.690210v2)

#### Dataset
__NYU Retinotopy Dataset__

Himmelberg, Marc M. and Kurzawski, Jan W. and Benson, Noah C. and Carrasco, Marisa and Winawer, Jonathan. "NYU Retinotopy dataset" 2021. [doi:10.18112/openneuro.ds003787.v1.0.1](https://github.com/OpenNeuroDatasets/ds003787.git)

## Load Software Tools

In [ ]:
%%capture 
!pip install nilearn nibabel numpy seaborn

In [ ]:
# we can use module to load deepretinotopy in a specific version
import module
import os
import nibabel as nib
import numpy as np
import matplotlib
from matplotlib.colors import ListedColormap
await module.load('deepretinotopy/1.0.18')
await module.list()

In [ ]:
# Request a freesurfer license and store it in your homedirectory. This is just an example - please replace with your license id:
!echo "fernanda.ribeiro@uni-giessen.de" > ~/.license
!echo "12345" >> ~/.license
!echo "*CxxxXXxxXXxx" >> ~/.license
!echo "FS/XXXXXXXXXX" >> ~/.license

## Visualizing ROIs

First, we will visualize some ROIs based on a [probabilistic atlas of visual areas](https://academic.oup.com/cercor/article-lookup/doi/10.1093/cercor/bhu277). We can start by visualizing these ROIs or parcels in the fs_32k_LR space as follows:

In [ ]:
# Set up paths relative to the current directory
os.environ['PATH_FREESURFER_DIR'] = "./data/ds003787/derivatives/freesurfer"
os.environ['PATH_HCP_TEMPLATE_SURFACE'] = "./data/templates"
os.environ['DATASET_NAME'] = "nyu"
os.environ['SUBJECT_ID'] = "sub-wlsubj001"

In [ ]:
from nilearn import plotting
from ipywidgets import Dropdown, interact
from IPython.display import display
import matplotlib

def plot_parcel(parcel, surface='sphere'):
    subject = os.environ['SUBJECT_ID']
    if surface == 'sphere':
        mesh_file = f"{os.environ['PATH_HCP_TEMPLATE_SURFACE']}/fs_LR-deformed_to-fsaverage.L.sphere.32k_fs_LR.surf.gii"
    elif surface == 'midthickness':
        mesh_file = f"{os.environ['PATH_FREESURFER_DIR']}/{subject}/surf/{subject}.lh.midthickness.32k_fs_LR.surf.gii"

    data = np.array(nib.load(f"../rois/{parcel}.lh.32k_fs_LR.label.gii").agg_data())[:]

    # Curvature as background
    background = np.array(nib.load(f"{os.environ['PATH_FREESURFER_DIR']}/{subject}/surf/{subject}.curvature-midthickness.lh.32k_fs_LR.func.gii").agg_data())[:]
    # binarizing curvature maps
    background[background <= 0] = -1
    background[background > 1] = 1
    # Select first 4 colours from tab20b (the purple shades) by index
    newcmp = ListedColormap(matplotlib.colormaps['tab20b'].colors[:4], name='VisualAreas')

    view = plotting.view_surf(
            surf_mesh= mesh_file,
            surf_map=np.reshape(data[:], (-1)),
            cmap=newcmp, black_bg=False, symmetric_cmap=False, bg_map = background,
            threshold=0.1, vmin = 1, vmax = 3)
    return view

In [ ]:
parcels = ["V1dorsal", "V1ventral", "V2dorsal", "V2ventral", "V123plusfovea"]
surfaces = ['sphere', 'midthickness']

@interact(parcel=Dropdown(options=parcels, description="Parcel:"), surface=surfaces)
def update(parcel, surface):
    return plot_parcel(parcel, surface)

## Register atlas to the fs native space

Before estimating cortical surface area, we need to resample the parcellation labels from the `32k_fs_LR` template space to each subject's native cortical surface. This is done using `wb_command -label-resample` with the `ADAP_BARY_AREA` method, which preserves the areal distribution of labels during resampling.

In [ ]:
import subprocess

for subject in ['sub-wlsubj001', 'sub-wlsubj004', 'sub-wlsubj006', 'sub-wlsubj007']:
    for hemisphere, hemi in [('lh', 'L'), ('rh', 'R')]:
        for parcel in ['V1dorsal', 'V1ventral', 'V2dorsal', 'V2ventral', 'V123plusfovea']:
            # subject = os.environ['SUBJECT_ID']
            fs_dir = os.environ['PATH_FREESURFER_DIR']
            template_dir = os.environ['PATH_HCP_TEMPLATE_SURFACE']
    
            # Disable Apptainer overlay to avoid nested container errors in JupyterHub
            env = os.environ.copy()
            env['APPTAINER_NO_MOUNT'] = 'all'
    
            cmd = (
                f"wb_command -label-resample "
                f"../rois/{parcel}.{hemisphere}.32k_fs_LR.label.gii "
                f"{template_dir}/fs_LR-deformed_to-fsaverage.{hemi}.sphere.32k_fs_LR.surf.gii "
                f"{fs_dir}/{subject}/surf/{hemisphere}.sphere.reg.surf.gii "
                f"ADAP_BARY_AREA "
                f"{fs_dir}/{subject}/deepRetinotopy/{subject}.{parcel}-roi.{hemisphere}.native.label.gii "
                f"-area-surfs "
                f"{fs_dir}/{subject}/surf/{subject}.{hemisphere}.midthickness.32k_fs_LR.surf.gii "
                f"{fs_dir}/{subject}/surf/{hemisphere}.midthickness.surf.gii"
            )
            result = subprocess.run(cmd, shell=True, env=env, capture_output=True, text=True)
            if result.returncode != 0:
                print(f"Error resampling {parcel} {hemisphere}:\n{result.stderr}")
            else:
                print(f"Done: {parcel} {hemisphere}")

## Estimating cortical surface area

Cortical surface area is estimated directly from the triangular mesh of the subject's midthickness surface (the surface halfway between the white matter and pial surfaces). The mesh is composed of vertices — points in 3D space — connected into triangles. For each triangle, its area is computed using the **cross product** of two edge vectors:

$$A_i = \frac{1}{2} \| \vec{v}_1 \times \vec{v}_2 \|$$

where $\vec{v}_1$ and $\vec{v}_2$ are vectors along two edges of the triangle. Only triangles whose **all three vertices** fall within the ROI mask are included in the sum, giving the total surface area of the region in mm².

This approach is applied to parcels resampled to the subject's native space, ensuring that area estimates reflect the individual's cortical geometry rather than a template.

In [ ]:
def compute_surface_area(vertices, faces, mask):
    """Compute surface area for masked region"""
    # Get triangles that have at least one vertex in the mask
    # Generate a (F, 3, 3) array — F triangles, each with 3 vertices, each vertex having 3 coordinates:
    triangles = vertices[faces]
    
    # Calculate triangle areas using cross product
    v1 = triangles[:, 1] - triangles[:, 0]
    v2 = triangles[:, 2] - triangles[:, 0]
    areas = 0.5 * np.linalg.norm(np.cross(v1, v2), axis=1)
    
    # Only include triangles where all vertices are in the mask
    valid_faces = np.all(mask[faces], axis=1)
    return np.sum(areas[valid_faces])

In [ ]:
import pandas as pd

subject = os.environ['SUBJECT_ID']
fs_dir  = os.environ['PATH_FREESURFER_DIR']

rows = []
for hemisphere in ['lh', 'rh']:
    surface = nib.load(f"{fs_dir}/{subject}/surf/{hemisphere}.midthickness.surf.gii")
    vertices = surface.darrays[0].data
    faces    = surface.darrays[1].data

    v123 = nib.load(f"{fs_dir}/{subject}/deepRetinotopy/{subject}.V123plusfovea-roi.{hemisphere}.native.label.gii").darrays[0].data

    for label, roi_name in [(1, 'V1'), (2, 'V2'), (3, 'V3')]:
        mask = v123 == label
        area = compute_surface_area(vertices, faces, mask)
        rows.append({'subject': subject, 'hemisphere': hemisphere, 'ROI': roi_name, 'surface_area_mm2': round(area, 2)})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

### Cortical Magnification Factor

Cortical magnification factor (CMF) describes how much cortical surface area (mm²) is dedicated to each degree² of visual space. It is computed from 1° to 7° of eccentricity. At each eccentricity value *r*, we find a window Δr such that exactly 20% of the ROI vertices have an eccentricity within [r − Δr, r + Δr]. The surface area of those vertices is then divided by the area of the corresponding visual field annulus:

$$m(r) = \frac{\text{surface area (mm}^2\text{)}}{\pi \left[(r + \Delta r)^2 - (r - \Delta r)^2\right]}$$

In [ ]:
def compute_cmf(vertices_lh, faces_lh, roi_mask_lh, eccentricity_lh,
                vertices_rh, faces_rh, roi_mask_rh, eccentricity_rh,
                eccentricities=np.arange(1, 8)):
    """Compute areal cortical magnification factor (mm²/deg²) for a given ROI.

    Surface areas from both hemispheres are summed and divided by the full annulus
    area, since together both hemispheres represent a complete annulus of visual space.
    Δr is found from the combined vertex pool across both hemispheres, such that 20%
    of all ROI vertices fall within [r - Δr, r + Δr].
    """
    valid_lh = roi_mask_lh & (eccentricity_lh > 0) & ~np.isnan(eccentricity_lh)
    valid_rh = roi_mask_rh & (eccentricity_rh > 0) & ~np.isnan(eccentricity_rh)

    ecc_combined = np.concatenate([eccentricity_lh[valid_lh], eccentricity_rh[valid_rh]])
    n_target = max(1, int(0.2 * len(ecc_combined)))

    cmf_values = []
    for r in eccentricities:
        distances = np.sort(np.abs(ecc_combined - r))
        delta_r = distances[n_target - 1]

        annulus_lh = valid_lh & (eccentricity_lh >= r - delta_r) & (eccentricity_lh <= r + delta_r)
        annulus_rh = valid_rh & (eccentricity_rh >= r - delta_r) & (eccentricity_rh <= r + delta_r)

        total_area  = (compute_surface_area(vertices_lh, faces_lh, annulus_lh) +
                       compute_surface_area(vertices_rh, faces_rh, annulus_rh))
        visual_space = np.pi * ((r + delta_r)**2 - (r - delta_r)**2)  # full annulus
        cmf_values.append(total_area / visual_space if visual_space > 0 else np.nan)

    return cmf_values


def compute_cmf_across_subjects(subjects):
    fs_dir         = os.environ['PATH_FREESURFER_DIR']
    eccentricities = np.arange(1, 8)
    rows = []

    for subject in subjects:
        # Load both hemispheres once per subject
        surfaces, faces_dict, eccentricities_dict, v123_dict = {}, {}, {}, {}
        for hemisphere in ['lh', 'rh']:
            surf = nib.load(f"{fs_dir}/{subject}/surf/{hemisphere}.midthickness.surf.gii")
            surfaces[hemisphere]      = surf.darrays[0].data
            faces_dict[hemisphere]    = surf.darrays[1].data
            eccentricities_dict[hemisphere] = nib.load(
                f"{fs_dir}/{subject}/deepRetinotopy/{subject}.predicted_eccentricity_model.{hemisphere}.native.func.gii"
            ).darrays[0].data
            v123_dict[hemisphere] = nib.load(
                f"{fs_dir}/{subject}/deepRetinotopy/{subject}.V123plusfovea-roi.{hemisphere}.native.label.gii"
            ).darrays[0].data

        for label, roi_name in [(1, 'V1'), (2, 'V2'), (3, 'V3')]:
            cmf_values = compute_cmf(
                surfaces['lh'], faces_dict['lh'], v123_dict['lh'] == label, eccentricities_dict['lh'],
                surfaces['rh'], faces_dict['rh'], v123_dict['rh'] == label, eccentricities_dict['rh'],
                eccentricities,
            )
            for r, cmf in zip(eccentricities, cmf_values):
                rows.append({'subject': subject, 'ROI': roi_name,
                             'eccentricity_deg': r, 'CMF_mm2_deg2': round(cmf, 4)})

    df_cmf = pd.DataFrame(rows)
    print(df_cmf.to_string(index=False))
    return df_cmf

In [ ]:
df_cmf = compute_cmf_across_subjects(['sub-wlsubj001', 'sub-wlsubj004', 'sub-wlsubj006', 'sub-wlsubj007'])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats

roi_colors = {'V1': '#3d0f8f', 'V2': '#6a3db5', 'V3': '#b89fde'}

fig, ax = plt.subplots(figsize=(6, 4))

for roi_name, color in roi_colors.items():
    df_roi = df_cmf[df_cmf['ROI'] == roi_name].groupby('eccentricity_deg')['CMF_mm2_deg2']
    means = df_roi.mean()
    n     = df_roi.count()
    ci    = df_roi.sem() * stats.t.ppf(0.975, df=n - 1)

    ax.plot(means.index, means.values, marker='o', color=color, label=roi_name)
    ax.fill_between(means.index, means - ci, means + ci, alpha=0.25, color=color)

ax.set_xlabel('Eccentricity (°)')
ax.set_ylabel('CMF (mm² / deg²)')
ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
ax.legend(title='ROI')
sns.despine()
plt.tight_layout()
plt.show()

## [Optional] Estimate visual field maps

The visual field sign analysis (Sereno et al., 1995; Sereno et al., 1994) combines both polar angle and eccentricity maps into a unique representation of the visual field as either a non-mirror-image or a mirror-image representation of the retina. From this binary representation, it is quite easy to determine boundaries between visual areas. Note that this feature is not yet well-integrated into the current toolbox.

In [ ]:
%%bash
for hemisphere in lh rh;
    do 
        4_signmaps.py --path $PATH_FREESURFER_DIR --hemisphere $hemisphere --subject_id $SUBJECT_ID; 
    done

In [ ]:
from nilearn import plotting
import nibabel as nib
import numpy as np

# Configuration
subject = os.environ['SUBJECT_ID']
mesh_file = f"{os.environ['PATH_HCP_TEMPLATE_SURFACE']}/fs_LR-deformed_to-fsaverage.L.sphere.32k_fs_LR.surf.gii"
data = np.array(nib.load(f"{os.environ['PATH_FREESURFER_DIR']}/{subject}/deepRetinotopy/{subject}.fs_predicted_fieldSignMap_lh_model.func.gii").agg_data())[:]

# The binary visual field sign map is represented as 1 and -1, and vertices without predictions as -10
# so we have to do a few tricks so we can plot it nicely using nilearn
data = data + 2
data[data == -8] = 0

# Curvature as background
background = np.array(nib.load(f"{os.environ['PATH_FREESURFER_DIR']}/{subject}/surf/{subject}.curvature-midthickness.lh.32k_fs_LR.func.gii").agg_data())[:]
# binarizing curvature maps
background[background <= 0] = -1.
background[background > 1] = 1.

plotting.view_surf(
        surf_mesh= mesh_file,
        surf_map=np.reshape(data[:], (-1)),
        cmap="gist_rainbow_r", black_bg=False, symmetric_cmap=False, bg_map = background,
        threshold=1)